IMAGE CLASSIFICATION (CIFAR-10)

In [3]:
# Imports
import tensorflow as tf 
import keras
from keras import layers, models 
import keras_tuner as kt
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time

sns.set_theme(style="darkgrid")

Loading data

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

#Normalize pixel values
x_train = x_train/255.0
x_test = x_test/255.0

print(f"Training data shape: {x_train.shape}\n")
print(f"Test data shape: {x_test.shape}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2855s 17us/step
Training data shape: (50000, 32, 32, 3)

Test data shape: (10000, 32, 32, 3)


Building baseline model

In [ ]:
def create_baseline_model():
    model = models.Sequential([
        Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model 

print("--- Running 3-Fold Cross Validation on Baseline Model ---")
kf = KFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = []
fold_nr = 1

for train_index, val_index in kf.split(x_train):
    x_train_fold, x_val_fold = x_train[train_index], x_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    baseline = create_baseline_model()
    print(f"Training fold {fold_nr}...")

    baseline.fit(x_train_fold, y_train_fold, epochs=3, batch_size=64, validation_data=(x_val_fold, y_val_fold), verbose=0)
    scores = baseline.evaluate(x_val_fold, y_val_fold, verbose=0)
    cv_scores.append(scores[1])
    fold_nr += 1

print(f"\nBaseline CV accuracy: {np.mean(cv_scores)*100:.2f}% (+/- {np.std(cv_scores)*100:.2f}%)")

--- Running 3-Fold Cross Validation on Baseline Model ---


c:\Users\Clopo\Desktop\Model_optimization\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training fold 1...
Training fold 2...
Training fold 3...

Baseline CV accuracy: 61.32% (+/- 0.57%)
